# Imports & Functions

In [40]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [41]:
def clean_csv(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        lines = [line.strip().split(",") for line in f]
    n_cols = max(len(line) for line in lines)
    normalized = [line + [""] * (n_cols - len(line)) if len(line) < n_cols else line[:n_cols] for line in lines]
    df = pd.DataFrame(normalized)
    df.replace('', np.nan, inplace=True)
    index_values = list(range(50, 1050, 50)) + list(range(2000, 10001, 2000))
    df.index = index_values
    df.index.name = "nbSim"
    return df

def compute_references(df: pd.DataFrame) -> pd.DataFrame:
    df_res = df.copy()
    df_res["mean"] = df.astype(float).mean(axis=1, skipna=True)
    df_res["var"] = df.astype(float).var(axis=1, skipna=True)
    df_res["var_normalized"] = df_res["var"] / df_res.index
    df_res["ci_margin"] = (df_res["var_normalized"] * 1.96)
    df_res["ci_lower"] = df_res["mean"] - df_res["ci_margin"]
    df_res["ci_upper"] = df_res["mean"] + df_res["ci_margin"]
    return df_res

def summarize_results(df_dict: dict[str, pd.DataFrame], idx: int) -> pd.DataFrame:
    summary = {}

    for name, df in df_dict.items():
        mean = df.loc[idx, "mean"]
        var = df.loc[idx, "var_normalized"]
        ci_str = f"[{df.loc[idx, 'ci_lower']:.4f} ; {df.loc[idx, 'ci_upper']:.4f}]"

        summary[name] = {
            "mean": mean,
            "var": var,
            "ci_interval": ci_str
        }

    return pd.DataFrame(summary)

# Monte Carlo Mono Underlying Call

In [42]:
call_mc_path = r"Outputs/MonteCarlo_Simulations_Convergence_Call.csv"
call_mc_cv_path = r"Outputs/MonteCarlo_ControlVariate_Simulations_Convergence_Call.csv"
call_mc_an_path = r"Outputs/MonteCarlo_Antithetic_Simulations_Convergence_Call.csv"
call_mc_an_cv_path = r"Outputs/MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Call.csv"

In [43]:
df_call_mc = clean_csv(call_mc_path)
df_call_mc = compute_references(df_call_mc)

df_call_mc_cv = clean_csv(call_mc_cv_path)
df_call_mc_cv = compute_references(df_call_mc_cv)

df_call_mc_an = clean_csv(call_mc_an_path)
df_call_mc_an = compute_references(df_call_mc_an)

df_call_mc_an_cv = clean_csv(call_mc_an_cv_path)
df_call_mc_an_cv = compute_references(df_call_mc_an_cv)

In [44]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_mc.index,
    y=df_call_mc["mean"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_cv.index,
    y=df_call_mc_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an.index,
    y=df_call_mc_an["mean"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an_cv.index,
    y=df_call_mc_an_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Price Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [45]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_mc.index,
    y=df_call_mc["var_normalized"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_cv.index,
    y=df_call_mc_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an.index,
    y=df_call_mc_an["var_normalized"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_call_mc_an_cv.index,
    y=df_call_mc_an_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Call Variance Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [46]:
df_dict = {
    "MonteCarlo_Simulations_Call": df_call_mc,
    "MonteCarlo_ControlVariate_Simulations_Call": df_call_mc_cv,
    "MonteCarlo_Antithetic_Simulations_Call": df_call_mc_an,
    "MonteCarlo_Antithetic_ControlVariate_Simulations_Call": df_call_mc_an_cv
}

df_summarize = summarize_results(df_dict, 750)
df_summarize

,MonteCarlo_Simulations_Call,MonteCarlo_ControlVariate_Simulations_Call,MonteCarlo_Antithetic_Simulations_Call,MonteCarlo_Antithetic_ControlVariate_Simulations_Call
mean,7.108469,6.80496,6.85432,6.80496
var,0.08154,0.0,0.086928,0.0
ci_interval,[6.9487 ; 7.2683],[6.8050 ; 6.8050],[6.6839 ; 7.0247],[6.8050 ; 6.8050]


# Monte Carlo Mono Underlying Put

In [47]:
put_mc_path = r"Outputs/MonteCarlo_Simulations_Convergence_Put.csv"
put_mc_cv_path = r"Outputs/MonteCarlo_ControlVariate_Simulations_Convergence_Put.csv"
put_mc_an_path = r"Outputs/MonteCarlo_Antithetic_Simulations_Convergence_Put.csv"
put_mc_an_cv_path = r"Outputs/MonteCarlo_Antithetic_ControlVariate_Simulations_Convergence_Put.csv"

In [48]:
df_put_mc = clean_csv(put_mc_path)
df_put_mc = compute_references(df_put_mc)

df_put_mc_cv = clean_csv(put_mc_cv_path)
df_put_mc_cv = compute_references(df_put_mc_cv)

df_put_mc_an = clean_csv(put_mc_an_path)
df_put_mc_an = compute_references(df_put_mc_an)

df_put_mc_an_cv = clean_csv(put_mc_an_cv_path)
df_put_mc_an_cv = compute_references(df_put_mc_an_cv)

In [49]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_mc.index,
    y=df_put_mc["mean"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_cv.index,
    y=df_put_mc_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an.index,
    y=df_put_mc_an["mean"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an_cv.index,
    y=df_put_mc_an_cv["mean"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Price Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [50]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_mc.index,
    y=df_put_mc["var_normalized"],
    mode='lines',
    name='Monte Carlo'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_cv.index,
    y=df_put_mc_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an.index,
    y=df_put_mc_an["var_normalized"],
    mode='lines',
    name='Monte Carlo Antithetic'
))

fig.add_trace(go.Scatter(
    x=df_put_mc_an_cv.index,
    y=df_put_mc_an_cv["var_normalized"],
    mode='lines',
    name='Monte Carlo Control Variate Antithetic'
))

fig.update_layout(
    title="Put Variance Convergence by method of Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

In [51]:
df_dict_put = {
    "MonteCarlo_Simulations_Put": df_put_mc,
    "MonteCarlo_ControlVariate_Simulations_Put": df_put_mc_cv,
    "MonteCarlo_Antithetic_Simulations_Put": df_put_mc_an,
    "MonteCarlo_Antithetic_ControlVariate_Simulations_Put": df_put_mc_an_cv
}

df_summarize_put = summarize_results(df_dict_put, 750)
df_summarize_put

,MonteCarlo_Simulations_Put,MonteCarlo_ControlVariate_Simulations_Put,MonteCarlo_Antithetic_Simulations_Put,MonteCarlo_Antithetic_ControlVariate_Simulations_Put
mean,2.255182,1.9279,1.980701,1.9279
var,0.021813,0.0,0.022397,0.0
ci_interval,[2.2124 ; 2.2979],[1.9279 ; 1.9279],[1.9368 ; 2.0246],[1.9279 ; 1.9279]


# Quasi Monte Carlo Mono Underlying Call

In [52]:
call_quasi_mc_path = r"Outputs/Quasi_MonteCarlo_Simulations_Convergence_Call.csv"

In [53]:
df_call_quasi_mc = clean_csv(call_quasi_mc_path)
df_call_quasi_mc = compute_references(df_call_quasi_mc)

In [54]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_quasi_mc.index,
    y=df_call_quasi_mc["mean"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.update_layout(
    title="Call Price Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [55]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_call_quasi_mc.index,
    y=df_call_quasi_mc["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.update_layout(
    title="Call Variance Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()

# Quasi Monte Carlo Mono Underlying Put

In [56]:
put_quasi_mc_path = r"Outputs/Quasi_MonteCarlo_Simulations_Convergence_Put.csv"

In [57]:
df_put_quasi_mc = clean_csv(put_quasi_mc_path)
df_put_quasi_mc = compute_references(df_put_quasi_mc)

In [58]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_quasi_mc.index,
    y=df_put_quasi_mc["mean"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.update_layout(
    title="Put Price Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Price",
    legend_title="Simulation methods"
)

fig.show()

In [59]:
fig = go.Figure()

# Courbe 1 : Moyenne
fig.add_trace(go.Scatter(
    x=df_put_quasi_mc.index,
    y=df_put_quasi_mc["var_normalized"],
    mode='lines',
    name='Quasi Monte Carlo'
))

fig.update_layout(
    title="Put Variance Convergence by method of Quasi Monte Carlo simulation",
    xaxis_title="Number of Simulations",
    yaxis_title="Average Variance",
    legend_title="Simulation methods"
)

fig.show()